# Financial Correlation Networks: Math to Code Walkthrough
This notebook maps the mathematical equations from the report directly to the Python implementation in the `src/finnet_backbone/` package.

## 1. Log-Returns and the Distance Metric
We calculate the log-returns $r_i(t) = \ln P_i(t) - \ln P_i(t-1)$ and convert the Pearson correlation matrix $C_{ij}$ into a valid metric space distance:
$$d_{ij} = \sqrt{2(1 - C_{ij})}$$
This is the Euclidean chord distance between standardized vectors on a unit hypersphere, guaranteeing it satisfies the triangle inequality.

In [ ]:
import numpy as np
import pandas as pd
from src.finnet_backbone.data_loader import load_market_data

# Load data and calculate returns
data, tickers, sectors, N, mapping = load_market_data()
log_returns = np.log(data / data.shift(1)).dropna()

# Calculate Correlation and Distance
rho_matrix = log_returns.corr(method="pearson")
rho_clipped = np.clip(rho_matrix.values, -1.0, 1.0)
distance_matrix = np.sqrt(2 * (1 - rho_clipped))

print(f"Distance Matrix Shape: {distance_matrix.shape}")
print(f"Min Distance: {distance_matrix.min():.4f}, Max Distance: {distance_matrix.max():.4f}")

## 2. Random Matrix Theory (RMT)
If the assets were uncorrelated, the eigenvalues of the correlation matrix would follow the Marčenko-Pastur distribution. The bounds for the noise bulk are:
$$\lambda_{\pm} = \sigma^2 \left(1 \pm \sqrt{\frac{1}{Q}}\right)^2$$
where $Q = T/N$. Eigenvalues above $\lambda_+$ represent real macroeconomic factors (sector modes), not noise.

In [ ]:
from src.finnet_backbone.rmt import mp_null_lambda_plus

# Calculate eigenvalues
eigvals, eigvecs = np.linalg.eigh(rho_matrix.values)
eigvals_sorted = eigvals[::-1]

# Remove market mode and calculate MP bounds
T_eff, N_eff = log_returns.shape
Q = T_eff / N_eff
lambda_minus = (1 - 1/np.sqrt(Q))**2
lambda_plus = (1 + 1/np.sqrt(Q))**2

print(f"Aspect Ratio Q: {Q:.2f}")
print(f"Theoretical Noise Bounds: λ- = {lambda_minus:.4f}, λ+ = {lambda_plus:.4f}")
print(f"Market Mode λ1: {eigvals_sorted[0]:.4f}")

## 3. The Disparity Filter
Global thresholding destroys local structure. The Disparity Filter evaluates edges locally. For a node $i$ with strength $s_i$ and degree $k_i$, the probability of an edge weight $W_{ij}$ occurring by chance is:
$$\alpha_{ij}^{(i)} = \left(1 - \frac{W_{ij}}{s_i}\right)^{k_i - 1}$$
We keep the edge if $p_{ij} = \min(\alpha_{ij}^{(i)}, \alpha_{ji}^{(j)}) < \alpha$.

In [ ]:
import networkx as nx
from src.finnet_backbone.backbone import disparity_filter_naive

# Apply Disparity Filter
W_pure = np.abs(rho_matrix.values)
np.fill_diagonal(W_pure, 0.0)
backbone_np, pvals = disparity_filter_naive(W_pure, alpha=0.05)

G_backbone = nx.from_numpy_array(backbone_np)
G_backbone = nx.relabel_nodes(G_backbone, mapping)
G_backbone.remove_nodes_from(list(nx.isolates(G_backbone)))

print(f"Disparity Backbone: {G_backbone.number_of_nodes()} nodes, {G_backbone.number_of_edges()} edges.")

## 4. Spectral Graph Theory and Robustness
We evaluate the network's connectivity using the Laplacian matrix $\mathbf{L} = \mathbf{D} - \mathbf{A}$. The second smallest eigenvalue $\mu_2$ (Fiedler value) dictates global connectivity. We also simulate targeted attacks to measure the decay of the Giant Connected Component (GCC).

In [ ]:
from src.finnet_backbone.backbone import safe_algebraic_connectivity, track_gcc_decay

# Calculate Fiedler Value
fiedler = safe_algebraic_connectivity(G_backbone)
print(f"Algebraic Connectivity (Fiedler λ2): {fiedler:.4f}")

# Simulate Targeted Attack
degrees = dict(G_backbone.degree())
targeted_order = sorted(degrees.keys(), key=lambda x: degrees[x], reverse=True)
gcc_decay = track_gcc_decay(G_backbone, targeted_order, N)

print(f"Initial GCC Size: {gcc_decay[0]*100:.2f}%")
print(f"GCC Size after removing top 3 hubs: {gcc_decay[3]*100:.2f}%")